In [ ]:
# GAN Image Generator using TensorFlow/Keras

# Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Step 1: Load and preprocess the dataset
(x_train, _), (_, _) = tf.keras.datasets.cifar10.load_data()

# Normalize the dataset to the range [-1, 1] for the generator's tanh activation
x_train = (x_train - 127.5) / 127.5

# Step 2: Define the generator model
def build_generator():
    """
    Builds the generator model for the GAN.
    The generator takes random noise as input and generates images.
    """
    model = tf.keras.Sequential([
        layers.Dense(256, input_dim=100),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(512),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(1024),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(32 * 32 * 3, activation='tanh'),
        layers.Reshape((32, 32, 3))
    ])
    return model

# Step 3: Define the discriminator model
def build_discriminator():
    """
    Builds the discriminator model for the GAN.
    The discriminator takes an image as input and outputs a probability of whether the image is real or fake.
    """
    model = tf.keras.Sequential([
        layers.Flatten(input_shape=(32, 32, 3)),
        layers.Dense(1024),
        layers.LeakyReLU(0.2),
        layers.Dense(512),
        layers.LeakyReLU(0.2),
        layers.Dense(256),
        layers.LeakyReLU(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Step 4: Build the GAN model
def build_gan(generator, discriminator):
    """
    Combines the generator and discriminator into a GAN model.
    The discriminator is frozen during the generator's training.
    """
    discriminator.trainable = False
    model = tf.keras.Sequential([generator, discriminator])
    return model

# Step 5: Compile the models
generator = build_generator()
discriminator = build_discriminator()
gan = build_gan(generator, discriminator)

discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Step 6: Train the GAN
def train_gan(epochs=10000, batch_size=64):
    """
    Trains the GAN by alternating between training the discriminator and the generator.
    """
    for epoch in range(epochs):
        # Generate random noise and create fake images
        noise = np.random.normal(0, 1, (batch_size, 100))
        generated_images = generator.predict(noise)
        
        # Select a random batch of real images
        real_images = x_train[np.random.randint(0, x_train.shape[0], batch_size)]
        
        # Create labels for real (1) and fake (0) images
        labels_real = np.ones((batch_size, 1))
        labels_fake = np.zeros((batch_size, 1))
        
        # Train the discriminator on real and fake images
        d_loss_real = discriminator.train_on_batch(real_images, labels_real)
        d_loss_fake = discriminator.train_on_batch(generated_images, labels_fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        # Train the generator (via the GAN model)
        noise = np.random.normal(0, 1, (batch_size, 100))
        valid_y = np.ones((batch_size, 1))  # The generator wants the discriminator to label its outputs as real
        g_loss = gan.train_on_batch(noise, valid_y)
        
        # Print progress every 100 epochs
        if epoch % 100 == 0:
            print(f"{epoch} [D loss: {d_loss[0]:.4f}, acc.: {100 * d_loss[1]:.2f}%] [G loss: {g_loss:.4f}]")
            
            # Save generated images for visualization
            if epoch % 1000 == 0:
                save_generated_images(epoch, generator)

# Step 7: Save generated images
def save_generated_images(epoch, generator, examples=25, dim=(5, 5), figsize=(10, 10)):
    """
    Saves a grid of generated images for visualization.
    """
    noise = np.random.normal(0, 1, (examples, 100))
    generated_images = generator.predict(noise)
    generated_images = 0.5 * generated_images + 0.5  # Rescale images to [0, 1]
    
    plt.figure(figsize=figsize)
    for i in range(examples):
        plt.subplot(dim[0], dim[1], i + 1)
        plt.imshow(generated_images[i])
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"gan_generated_image_epoch_{epoch}.png")
    plt.close()

# Train the GAN
train_gan(epochs=10000, batch_size=64)